# 01 — Recolha dos Dados

## Pergunta de investigação

> **Que dados temos?**

Neste notebook será realizada a identificação e recolha dos dados necessários para o projeto Construct Price Forecast.

### Objetivos

- Identificar as fontes de dados;
- Recolher os dados históricos;
- Conhecer a estrutura dos dados;
- Identificar os índices de custos de materiais disponíveis;
- Avaliar a qualidade e disponibilidade dos dados.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

In [3]:
PROJECT_ROOT = Path("..")

DATA_RAW = PROJECT_ROOT / "data" / "raw" / "impic"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Dados raw:", DATA_RAW)
print("Dados processados:", DATA_PROCESSED)

Dados raw: ..\data\raw\impic
Dados processados: ..\data\processed


## Índices de materiais do IMPIC

O IMPIC disponibiliza séries históricas de índices de custos de materiais
utilizados na revisão de preços de empreitadas.

A fonte oficial será utilizada como origem dos dados deste projeto.

In [4]:
IMPIC_URL = "https://www.impic.pt/impic/indices-de-revisao-de-precos-em-empreitadas/publicacao-em-diario-da-republica"

print("Fonte:", IMPIC_URL)

Fonte: https://www.impic.pt/impic/indices-de-revisao-de-precos-em-empreitadas/publicacao-em-diario-da-republica


## Recolha dos ficheiros históricos

O objetivo desta etapa é identificar e recolher os ficheiros históricos de
índices de custos de materiais disponibilizados pelo IMPIC.

Os ficheiros serão armazenados na pasta `data/raw/impic`, preservando os
dados originais antes de qualquer transformação.

A partir desses ficheiros será construído um catálogo histórico dos códigos
de materiais identificados.

In [5]:
import requests
from bs4 import BeautifulSoup

In [62]:
response = requests.get(IMPIC_URL)

response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

links = soup.find_all("a")

material_links = []

for link in links:
    href = link.get("href", "")
    text = link.get_text(" ", strip=True)

    if "materiais" in text.lower():
        material_links.append({
            "text": text,
            "url": href
        })

len(material_links)

35

In [80]:
pd.DataFrame(material_links)

,text,url
0,"Índices de custos de mão-de-obra, de materiais...",https://www.impic.pt/impic/pt-pt/indices-de-re...
1,Índices de custos de materiais - M01 a M41 - B...,https://www.impic.pt/impic/assets/misc/indices...
2,Mapa Global de Índices de custos de mão-de-obr...,https://www.impic.pt/impic/assets/misc/indices...
3,Índices de custos de materiais - M01 a M41 - B...,https://www.impic.pt/impic/assets/misc/indices...
4,Mapa Global de Índices de custos de mão-de-obr...,https://www.impic.pt/impic/assets/misc/indices...
5,Índices de custos de materiais - M01 a M41 - B...,https://www.impic.pt/impic/assets/misc/indices...
6,Mapa Global de Índices de custos de mão-de-obr...,https://www.impic.pt/impic/assets/misc/indices...
7,Índices de custos de materiais - M01 a M41 - B...,https://www.impic.pt/impic/assets/misc/indices...
8,2023: Mapa Global de Índices de custos de mão-...,https://www.impic.pt/impic/assets/misc/indices...
9,Índices de custos de materiais - M01 a M41 - B...,https://www.impic.pt/impic/assets/misc/indices...


## Recolha dos documentos históricos

Os documentos disponibilizados pelo IMPIC serão utilizados para construir
um catálogo histórico dos índices de materiais.

O objetivo é identificar todos os códigos de materiais que aparecem nas
diferentes publicações e preservar a sua evolução temporal.

In [81]:
# Remover URLs duplicados mantendo a ordem original

urls = list(dict.fromkeys(
    item["url"] for item in material_links
))

print(f"Links encontrados: {len(material_links)}")
print(f"Links únicos: {len(urls)}")

Links encontrados: 35
Links únicos: 35


In [82]:
# Resumo das publicações encontradas

for i, url in enumerate(urls, start=1):
    print(f"{i:02d} - {url}")

01 - https://www.impic.pt/impic/pt-pt/indices-de-revisao-de-precos-em-empreitadas/indices-de-custos-de-mao-de-obra-de-materiais-e-de-equipamentos-de-apoio
02 - https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2026_MATERIAIS.xls
03 - https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2026_.xls
04 - https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2025_MATERIAIS.xls
05 - https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2025_.xls
06 - https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2024_MATERIAIS.xls
07 - https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2024_.xls
08 - https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2023_MATERIAIS.xls
09 - https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2023_.xls
10 - https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2022_MATERIAIS.xls
11 - https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2022_.xls
12 - https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2021_MATERIAIS.xls
13 - https://www.impi

In [83]:
from urllib.parse import urlparse

# Classificar os recursos disponibilizados pelo IMPIC

material_files = [
    url for url in urls
    if "_MATERIAIS.XLS" in url.upper()
]

global_urls = [
    url for url in urls
    if "_MATERIAIS.XLS" not in url.upper()
    and url.lower().endswith(".xls")
]

pages = [
    url for url in urls
    if not url.lower().endswith(".xls")
]

print("Ficheiros de materiais:", len(material_files))
print("Ficheiros globais:", len(global_urls))
print("Páginas:", len(pages))

Ficheiros de materiais: 12
Ficheiros globais: 22
Páginas: 1


In [84]:
print("\n--- MATERIAIS ---")

for url in material_files:
    print(url)

print("\n--- GLOBAIS ---")

for url in global_urls:
    print(url)

print("\n--- PÁGINAS ---")

for url in pages:
    print(url)


--- MATERIAIS ---
https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2026_MATERIAIS.xls
https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2025_MATERIAIS.xls
https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2024_MATERIAIS.xls
https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2023_MATERIAIS.xls
https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2022_MATERIAIS.xls
https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2021_MATERIAIS.xls
https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2020_MATERIAIS.xls
https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2019_MATERIAIS.xls
https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2018_MATERIAIS.xls
https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2017_MATERIAIS.xls
https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2016_MATERIAIS.xls
https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2015_MATERIAIS.xls

--- GLOBAIS ---
https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2026_.xls
https://www

## Recolha dos ficheiros de materiais

Os ficheiros `CIFE*_MATERIAIS.xls` constituem a fonte principal
para a construção da série histórica dos índices de materiais.

Os ficheiros serão armazenados na área `data/raw/impic`,
mantendo os dados originais sem transformação.

In [85]:
import requests
from pathlib import Path

PDF_RAW = DATA_RAW  # mantemos o nome geral da pasta raw

for i, url in enumerate(material_files, start=1):

    filename = Path(url).name
    filepath = PDF_RAW / filename

    print(f"{i:02d}/{len(material_files)} - {filename}")

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    filepath.write_bytes(response.content)

print("\nDownload dos ficheiros de materiais concluído.")

01/12 - cife2026_MATERIAIS.xls
02/12 - cife2025_MATERIAIS.xls
03/12 - cife2024_MATERIAIS.xls
04/12 - cife2023_MATERIAIS.xls
05/12 - cife2022_MATERIAIS.xls
06/12 - cife2021_MATERIAIS.xls
07/12 - cife2020_MATERIAIS.xls
08/12 - cife2019_MATERIAIS.xls
09/12 - cife2018_MATERIAIS.xls
10/12 - cife2017_MATERIAIS.xls
11/12 - cife2016_MATERIAIS.xls
12/12 - cife2015_MATERIAIS.xls

Download dos ficheiros de materiais concluído.


In [16]:
%pip install xlrd

Note: you may need to restart the kernel to use updated packages.


In [86]:
pip show xlrd

Name: xlrd
Version: 2.0.2
Summary: Library for developers to extract data from Microsoft Excel (tm) .xls spreadsheet files
Home-page: http://www.python-excel.org/
Author: Chris Withers
Author-email: chris@withers.org
License: BSD
Location: c:\Users\nunon\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages
Requires: 
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [18]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [87]:
# ============================================================
# Inspeção dos ficheiros históricos do IMPIC
# ============================================================

import pandas as pd
from pathlib import Path

print("Ficheiros encontrados:\n")

for filepath in sorted(PDF_RAW.glob("*_MATERIAIS.xls")):

    print(f"\n{'=' * 70}")
    print(filepath.name)

    try:
        excel = pd.ExcelFile(filepath)

        print("Folhas:", excel.sheet_names)

        for sheet in excel.sheet_names:
            df = pd.read_excel(filepath, sheet_name=sheet)

            print(f"\n  Folha: {sheet}")
            print(f"  Dimensão: {df.shape}")
            print(f"  Colunas: {list(df.columns)}")

            display(df.head(5))

    except Exception as e:
        print(f"ERRO: {e}")

Ficheiros encontrados:


cife2015_MATERIAIS.xls
Folhas: ['2015-Mat-base100 Dez91;Jan04']

  Folha: 2015-Mat-base100 Dez91;Jan04
  Dimensão: (60, 14)
  Colunas: ['Unnamed: 0', 'Unnamed: 1', 'QUADRO II', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']


,Unnamed: 0,Unnamed: 1,QUADRO II,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,NaN,NaN,ÍNDICES CUSTOS DE MATERIAIS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,M01 a M41 - BASE 100: DEZEMBRO DE 1991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,M42 a M51 - BASE 100: JANEIRO DE 2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,CÓDIGO,MATERIAIS,ANO DE 2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,JAN,FEV,MAR,ABR,MAI,JUN,JUL,AGO,SET,OUT,NOV,DEZ



cife2016_MATERIAIS.xls
Folhas: ['2016-Mat-base100 Dez91;Jan04']

  Folha: 2016-Mat-base100 Dez91;Jan04
  Dimensão: (60, 14)
  Colunas: ['Unnamed: 0', 'Unnamed: 1', 'QUADRO II', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']


,Unnamed: 0,Unnamed: 1,QUADRO II,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,NaN,NaN,ÍNDICES CUSTOS DE MATERIAIS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,M01 a M41 - BASE 100: DEZEMBRO DE 1991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,M42 a M51 - BASE 100: JANEIRO DE 2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,CÓDIGO,MATERIAIS,ANO DE 2016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,JAN,FEV,MAR,ABR,MAI,JUN,JUL,AGO,SET,OUT,NOV,DEZ



cife2017_MATERIAIS.xls
Folhas: ['2017-Mat-base100 Dez91;Jan04']

  Folha: 2017-Mat-base100 Dez91;Jan04
  Dimensão: (60, 14)
  Colunas: ['QUADRO II', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']


...

,QUADRO II,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,ÍNDICES CUSTOS DE MATERIAIS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M01 a M41 - BASE 100: DEZEMBRO DE 1991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,M42 a M51 - BASE 100: JANEIRO DE 2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,M52 a M57 - BASE 100: DEZEMBRO DE 2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,M58 - BASE 100: DEZEMBRO DE 2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



cife2025_MATERIAIS.xls
Folhas: ['25-Mat-base100Dez91;Jan04;Dez21']

  Folha: 25-Mat-base100Dez91;Jan04;Dez21
  Dimensão: (69, 14)
  Colunas: ['QUADRO II', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']


,QUADRO II,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,ÍNDICES CUSTOS DE MATERIAIS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M01 a M41 - BASE 100: DEZEMBRO DE 1991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,M42 a M51 - BASE 100: JANEIRO DE 2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,M52 a M57 - BASE 100: DEZEMBRO DE 2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,M58 - BASE 100: DEZEMBRO DE 2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



cife2026_MATERIAIS.xls
Folhas: ['26-Mat-base100Dez91;Jan04;Dez21']

  Folha: 26-Mat-base100Dez91;Jan04;Dez21
  Dimensão: (72, 14)
  Colunas: ['QUADRO II', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']


,QUADRO II,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,ÍNDICES CUSTOS DE MATERIAIS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M01 a M41 - BASE 100: DEZEMBRO DE 1991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,M42 a M51 - BASE 100: JANEIRO DE 2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,M52 a M57 - BASE 100: DEZEMBRO DE 2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,M58 - BASE 100: DEZEMBRO DE 2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [88]:
for filepath in sorted(DATA_RAW.glob("*_MATERIAIS.xls")):

    print("\n" + "=" * 80)
    print(filepath.name)

    df = pd.read_excel(filepath, header=None)

    print("Dimensão:", df.shape)

    # Mostrar as primeiras 12 linhas
    display(df.iloc[:12, :15])


cife2015_MATERIAIS.xls
Dimensão: (61, 14)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,NaN,NaN,QUADRO II,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,ÍNDICES CUSTOS DE MATERIAIS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,M01 a M41 - BASE 100: DEZEMBRO DE 1991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,M42 a M51 - BASE 100: JANEIRO DE 2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CÓDIGO,MATERIAIS,ANO DE 2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,JAN,FEV,MAR,ABR,MAI,JUN,JUL,AGO,SET,OUT,NOV,DEZ
6,M01,Britas,104.4,104.7,105,104.8,104.5,104.4,104.4,105,105.7,104.034222,104.638252,104.638252
7,M02,Areias,87.9,87.9,87.9,87.9,87.9,87.9,87.9,87.9,87.9,87.911516,87.911516,87.911516
8,M03,Inertes,98.4,98.5,98.7,98.6,98.5,98.4,98.4,98.7,99,98.208919,98.487237,98.487237
9,M04,Ladrilhos de calcário e granito,96.1,96.1,96.1,96.1,96.1,96.1,96.1,96.1,96.1,96.096899,96.096899,96.096899



cife2016_MATERIAIS.xls
Dimensão: (61, 14)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,NaN,NaN,QUADRO II,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,ÍNDICES CUSTOS DE MATERIAIS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,M01 a M41 - BASE 100: DEZEMBRO DE 1991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,M42 a M51 - BASE 100: JANEIRO DE 2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CÓDIGO,MATERIAIS,ANO DE 2016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,JAN,FEV,MAR,ABR,MAI,JUN,JUL,AGO,SET,OUT,NOV,DEZ
6,M01,Britas,104.3,104.6,104.7,104.8,104.9,105.1,104.8,104.5,104.8,105.2,105.5,105.5
7,M02,Areias,87.9,87.9,87.9,87.9,87.9,87.9,87.9,87.9,87.9,87.9,87.9,87.9
8,M03,Inertes,98.3,98.5,98.5,98.6,98.6,98.7,98.6,98.4,98.6,98.8,98.9,98.9
9,M04,Ladrilhos de calcário e granito,96.1,96.1,96.1,96.1,96.1,96.1,96.1,96.1,96.1,96.1,96.1,96.1



cife2017_MATERIAIS.xls
Dimensão: (61, 14)


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,QUADRO II,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ÍNDICES CUSTOS DE MATERIAIS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,M01 a M41 - BASE 100: DEZEMBRO DE 1991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,M42 a M51 - BASE 100: JANEIRO DE 2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,M52 a M57 - BASE 100: DEZEMBRO DE 2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,M58 - BASE 100: DEZEMBRO DE 2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,CÓDIGO,MATERIAIS,ANO DE 2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,JAN,FEV,MAR,ABR,MAI,JUN,JUL,AGO,SET,OUT,NOV,DEZ
8,M01,Britas,144.7,147.5,147.5,147.5,147.7,148,147.1,147.6,149,147.3,147.4,147.4
9,M02,Areias,112.1,111,116.8,115.3,116.9,116.5,116.7,117.2,110.8,109.4,109.5,109.4



cife2025_MATERIAIS.xls
Dimensão: (70, 14)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,QUADRO II,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ÍNDICES CUSTOS DE MATERIAIS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,M01 a M41 - BASE 100: DEZEMBRO DE 1991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,M42 a M51 - BASE 100: JANEIRO DE 2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,M52 a M57 - BASE 100: DEZEMBRO DE 2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,M58 - BASE 100: DEZEMBRO DE 2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,CÓDIGO,MATERIAIS,ANO DE 2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,JAN,FEV,MAR,ABR,MAI,JUN,JUL,AGO,SET,OUT,NOV,DEZ
8,M01,Britas,149.1,150.5,150.7,150.9,151.3,151.7,152.2,153,151.1,151.8,151.2,152.2
9,M02,Areias,107.7,107.8,107.9,110.7,111.9,114.4,111.9,114.3,114.1,114,114.1,114.2



cife2026_MATERIAIS.xls
Dimensão: (73, 14)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,QUADRO II,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ÍNDICES CUSTOS DE MATERIAIS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,M01 a M41 - BASE 100: DEZEMBRO DE 1991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,M42 a M51 - BASE 100: JANEIRO DE 2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,M52 a M57 - BASE 100: DEZEMBRO DE 2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,M58 - BASE 100: DEZEMBRO DE 2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,M59 a M60 - BASE 100: DEZEMBRO DE 2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,CÓDIGO,MATERIAIS,ANO DE 2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,JAN,FEV,MAR,ABR,MAI,JUN,JUL,AGO,SET,OUT,NOV,DEZ
9,M01,Britas,160.5,161.7,162,163.8,165.6,165.7,167,NaN,NaN,NaN,NaN,NaN


In [89]:
import pandas as pd
from pathlib import Path


def parse_impic_materials(filepath):
    """
    Lê um ficheiro histórico de índices de materiais do IMPIC.

    Retorna um DataFrame com:
        ano
        codigo
        material
        mes
        indice
        ficheiro
    """

    filepath = Path(filepath)

    # ---------------------------------------------------------
    # 1. Ler o Excel sem assumir cabeçalho
    # ---------------------------------------------------------
    df = pd.read_excel(filepath, header=None)

    # ---------------------------------------------------------
    # 2. Descobrir a linha onde está CÓDIGO / MATERIAIS
    # ---------------------------------------------------------
    header_row = None

    for i, row in df.iterrows():
        values = row.astype(str).str.strip().str.upper().tolist()

        if "CÓDIGO" in values and "MATERIAIS" in values:
            header_row = i
            break

    if header_row is None:
        raise ValueError(
            f"Não foi encontrada a linha CÓDIGO/MATERIAIS em {filepath.name}"
        )

    # ---------------------------------------------------------
    # 3. A linha seguinte deverá conter JAN ... DEZ
    # ---------------------------------------------------------
    months_row = header_row + 1

    months = []

    for value in df.iloc[months_row]:
        value = str(value).strip().upper()

        if value in [
            "JAN", "FEV", "MAR", "ABR", "MAI", "JUN",
            "JUL", "AGO", "SET", "OUT", "NOV", "DEZ"
        ]:
            months.append(value)

    if len(months) != 12:
        raise ValueError(
            f"Foram encontrados {len(months)} meses em {filepath.name}: {months}"
        )

    # ---------------------------------------------------------
    # 4. Obter o ano a partir do nome do ficheiro
    # ---------------------------------------------------------
    # Exemplo:
    # cife2015_MATERIAIS.xls -> 2015

    import re

    match = re.search(r"cife(\d{4})", filepath.name.lower())

    if not match:
        raise ValueError(
            f"Não foi possível determinar o ano através do nome: {filepath.name}"
        )

    ano = int(match.group(1))

    # ---------------------------------------------------------
    # 5. Determinar as colunas dos meses
    # ---------------------------------------------------------
    month_columns = {}

    for col in df.columns:

        value = str(df.iloc[months_row, col]).strip().upper()

        if value in months:
            month_columns[value] = col

    # ---------------------------------------------------------
    # 6. Colunas do código e descrição
    # ---------------------------------------------------------
    code_col = None
    material_col = None

    for col in df.columns:

        value = str(df.iloc[header_row, col]).strip().upper()

        if value == "CÓDIGO":
            code_col = col

        elif value == "MATERIAIS":
            material_col = col

    if code_col is None or material_col is None:
        raise ValueError(
            f"Não foi possível identificar CÓDIGO/MATERIAIS em {filepath.name}"
        )

    # ---------------------------------------------------------
    # 7. Extrair os dados
    # ---------------------------------------------------------
    records = []

    for row_idx in range(months_row + 1, len(df)):

        codigo = df.iloc[row_idx, code_col]
        material = df.iloc[row_idx, material_col]

        # Ignorar linhas vazias
        if pd.isna(codigo):
            continue

        codigo = str(codigo).strip()

        # Só queremos códigos do tipo M01, M02 ... M57
        if not re.fullmatch(r"M\d{2}", codigo.upper()):
            continue

        material = str(material).strip()

        # -----------------------------------------------------
        # Criar um registo por mês
        # -----------------------------------------------------
        for mes, col in month_columns.items():

            valor = df.iloc[row_idx, col]

            # Converter valores numéricos
            valor = pd.to_numeric(valor, errors="coerce")

            records.append({
                "ano": ano,
                "codigo": codigo.upper(),
                "material": material,
                "mes": mes,
                "indice": valor,
                "ficheiro": filepath.name
            })

    return pd.DataFrame(records)

In [90]:
ficheiro_2015 = DATA_RAW / "cife2015_MATERIAIS.xls"

df_2015 = parse_impic_materials(ficheiro_2015)

print("Registos:", len(df_2015))

display(df_2015.head(20))

Registos: 588


,ano,codigo,material,mes,indice,ficheiro
0,2015,M01,Britas,JAN,104.400000,cife2015_MATERIAIS.xls
1,2015,M01,Britas,FEV,104.700000,cife2015_MATERIAIS.xls
2,2015,M01,Britas,MAR,105.000000,cife2015_MATERIAIS.xls
3,2015,M01,Britas,ABR,104.800000,cife2015_MATERIAIS.xls
4,2015,M01,Britas,MAI,104.500000,cife2015_MATERIAIS.xls
5,2015,M01,Britas,JUN,104.400000,cife2015_MATERIAIS.xls
6,2015,M01,Britas,JUL,104.400000,cife2015_MATERIAIS.xls
7,2015,M01,Britas,AGO,105.000000,cife2015_MATERIAIS.xls
8,2015,M01,Britas,SET,105.700000,cife2015_MATERIAIS.xls
9,2015,M01,Britas,OUT,104.034222,cife2015_MATERIAIS.xls


In [91]:
all_materials = []

for filepath in sorted(DATA_RAW.glob("*_MATERIAIS.xls")):

    print(f"Processando: {filepath.name}")

    try:
        df = parse_impic_materials(filepath)

        all_materials.append(df)

        print(f"  → {len(df)} registos")

    except Exception as e:
        print(f"  ❌ ERRO: {e}")


materials_history = pd.concat(
    all_materials,
    ignore_index=True
)

print()
print("=" * 60)
print("TOTAL")
print("=" * 60)

print("Registos:", len(materials_history))
print("Anos:", sorted(materials_history["ano"].unique()))
print("Códigos:", materials_history["codigo"].nunique())

Processando: cife2015_MATERIAIS.xls
  → 588 registos
Processando: cife2016_MATERIAIS.xls
  → 588 registos
Processando: cife2017_MATERIAIS.xls
  → 588 registos
Processando: cife2018_MATERIAIS.xls
  → 588 registos
Processando: cife2019_MATERIAIS.xls
  → 588 registos
Processando: cife2020_MATERIAIS.xls
  → 660 registos
Processando: cife2021_MATERIAIS.xls
  → 660 registos
Processando: cife2022_MATERIAIS.xls
  → 672 registos
Processando: cife2023_MATERIAIS.xls
  → 672 registos
Processando: cife2024_MATERIAIS.xls
  → 672 registos
Processando: cife2025_MATERIAIS.xls
  → 672 registos
Processando: cife2026_MATERIAIS.xls
  → 696 registos

TOTAL
Registos: 7644
Anos: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Códigos: 58


In [92]:
codes_by_year = (
    materials_history
    .groupby("ano")["codigo"]
    .unique()
)

for ano, codes in codes_by_year.items():

    print(
        f"{ano}: "
        f"{len(codes)} códigos | "
        f"{', '.join(sorted(codes))}"
    )

2015: 49 códigos | M01, M02, M03, M04, M05, M06, M07, M08, M09, M10, M12, M13, M14, M15, M16, M17, M18, M19, M20, M21, M22, M23, M24, M25, M26, M27, M28, M29, M30, M31, M32, M33, M34, M35, M36, M37, M39, M40, M41, M42, M43, M44, M45, M46, M47, M48, M49, M50, M51
2016: 49 códigos | M01, M02, M03, M04, M05, M06, M07, M08, M09, M10, M12, M13, M14, M15, M16, M17, M18, M19, M20, M21, M22, M23, M24, M25, M26, M27, M28, M29, M30, M31, M32, M33, M34, M35, M36, M37, M39, M40, M41, M42, M43, M44, M45, M46, M47, M48, M49, M50, M51
2017: 49 códigos | M01, M02, M03, M04, M05, M06, M07, M08, M09, M10, M12, M13, M14, M15, M16, M17, M18, M19, M20, M21, M22, M23, M24, M25, M26, M27, M28, M29, M30, M31, M32, M33, M34, M35, M36, M37, M39, M40, M41, M42, M43, M44, M45, M46, M47, M48, M49, M50, M51
2018: 49 códigos | M01, M02, M03, M04, M05, M06, M07, M08, M09, M10, M12, M13, M14, M15, M16, M17, M18, M19, M20, M21, M22, M23, M24, M25, M26, M27, M28, M29, M30, M31, M32, M33, M34, M35, M36, M37, M39, M40, M4

In [93]:
codes_by_year = (
    materials_history
    .groupby("ano")["codigo"]
    .unique()
)

In [94]:
m11 = materials_history[
    materials_history["codigo"] == "M11"
]

print("Registos M11:", len(m11))
print("Anos em que aparece:", sorted(m11["ano"].unique()))

m11

Registos M11: 0
Anos em que aparece: []


,ano,codigo,material,mes,indice,ficheiro


In [95]:
m11.groupby("ano").agg(
    meses=("mes", "count"),
    indice_min=("indice", "min"),
    indice_max=("indice", "max")
)

,meses,indice_min,indice_max
ano,,,


In [96]:
(
    materials_history[
        materials_history["codigo"].isin(["M10", "M11", "M12"])
    ]
    .groupby(["ano", "codigo"])
    .size()
    .unstack(fill_value=0)
)

codigo,M10,M12
ano,,
2015,12,12
2016,12,12
2017,12,12
2018,12,12
2019,12,12
2020,12,12
2021,12,12
2022,12,12
2023,12,12


In [97]:
materials_history[
    materials_history["codigo"]
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("M11")
].shape

(0, 6)

In [98]:
for ano, grupo in materials_history.groupby("ano"):
    
    codigos = set(
        grupo["codigo"]
        .astype(str)
        .str.strip()
        .str.upper()
    )
    
    esperados = {f"M{i:02d}" for i in range(1, 59)}
    faltam = sorted(esperados - codigos)
    
    print(f"{ano}: faltam {len(faltam)} → {', '.join(faltam)}")

2015: faltam 9 → M11, M38, M52, M53, M54, M55, M56, M57, M58
2016: faltam 9 → M11, M38, M52, M53, M54, M55, M56, M57, M58
2017: faltam 9 → M11, M38, M52, M53, M54, M55, M56, M57, M58
2018: faltam 9 → M11, M38, M52, M53, M54, M55, M56, M57, M58
2019: faltam 9 → M11, M38, M52, M53, M54, M55, M56, M57, M58
2020: faltam 3 → M11, M38, M58
2021: faltam 3 → M11, M38, M58
2022: faltam 2 → M11, M38
2023: faltam 2 → M11, M38
2024: faltam 2 → M11, M38
2025: faltam 2 → M11, M38
2026: faltam 2 → M11, M38


## Validação da cobertura histórica dos códigos de materiais

Após a recolha e normalização dos ficheiros históricos de índices de materiais
disponibilizados pelo IMPIC, foi realizada uma análise adicional à cobertura
dos códigos de materiais ao longo dos anos.

Esta análise surgiu da necessidade de distinguir entre:

- um código que não apresenta valores num determinado período;
- um código que apresenta valores mas não sofre variação;
- um código que não está presente no ficheiro publicado para esse período;
- um código que poderá ter sido introduzido ou descontinuado ao longo da série histórica.

### 1. Verificação específica do código M11

Foi realizada uma pesquisa direta pelo código `M11` no dataset histórico consolidado.

A pesquisa foi também normalizada através de:

- conversão para texto;
- remoção de espaços;
- conversão para maiúsculas.

O resultado foi:

- **0 registos encontrados para M11**;
- **nenhum ano entre 2015 e 2026 contém o código M11** nos ficheiros de materiais recolhidos.

Este resultado indica que, nos ficheiros analisados, a ausência do M11
não corresponde simplesmente à ausência de valores mensais ou à inexistência
de variação do índice. O código não está presente nos registos recolhidos.

### 2. Comparação com códigos adjacentes

Para verificar se esta ausência poderia resultar de um problema de leitura dos
ficheiros, foram comparados códigos próximos, nomeadamente `M10`, `M11` e `M12`.

Os resultados mostram que:

- `M10` possui 12 registos mensais por ano;
- `M12` possui 12 registos mensais por ano;
- `M11` não possui qualquer registo.

Esta comparação permite excluir, à partida, a hipótese de que todos os códigos
da zona M10–M12 estejam a ser ignorados pelo parser.

### 3. Análise da cobertura dos códigos por ano

Foi posteriormente calculado o conjunto de códigos existentes em cada ano.

A análise revelou uma evolução do número de códigos publicados:

- 2015–2019: 49 códigos;
- 2020: 55 códigos;
- 2021: 56 códigos;
- 2022–2026: 56/58 códigos, de acordo com os ficheiros recolhidos.

Também foram identificados códigos ausentes em cada período.

Entre os códigos que não aparecem na série analisada encontra-se
consistentemente o `M11`.

### 4. Interpretação

A análise demonstra que a composição dos códigos de materiais não é constante
ao longo de toda a série histórica.

Existem códigos que aparecem apenas a partir de determinados anos e outros
que não estão presentes nos ficheiros analisados.

É importante, contudo, não interpretar automaticamente a ausência de um código
como prova de que o material nunca existiu ou de que foi oficialmente
descontinuado.

Neste momento, a conclusão suportada pelos dados é mais restrita:

> **O código M11 não foi encontrado nos ficheiros históricos de índices de
> materiais do IMPIC recolhidos para o período 2015–2026.**

A eventual existência do código M11 em períodos anteriores, noutras
publicações, noutras séries de índices ou numa classificação diferente deverá
ser investigada através das fontes históricas correspondentes.

### 5. Nota metodológica

A comparação com a sequência `M01`–`M58` deve ser entendida apenas como um
teste de cobertura e não como uma lista oficial de códigos esperados.

A existência de um número de código na sequência não constitui, por si só,
evidência de que esse código tenha sido publicado pelo IMPIC em todos os anos.

Esta distinção será importante para a construção do catálogo histórico de
materiais e para evitar a criação artificial de séries temporais inexistentes.

### Conclusão desta etapa

O dataset histórico consolidado passa assim a permitir distinguir entre:

1. **código existente com valores mensais**;
2. **código existente apenas em determinados períodos**;
3. **código ausente de uma publicação/ano**;
4. **código cuja existência histórica ainda necessita de confirmação documental**.

Esta informação será preservada no catálogo de materiais e será utilizada nas
etapas seguintes de preparação dos dados e análise exploratória.

In [99]:
# Catálogo histórico dos materiais

materials_catalog = (
    materials_history
    .groupby(["codigo", "material"])
    .agg(
        primeiro_ano=("ano", "min"),
        ultimo_ano=("ano", "max"),
        numero_anos=("ano", "nunique"),
        numero_registos=("codigo", "size")
    )
    .reset_index()
    .sort_values(["codigo", "primeiro_ano"])
)

materials_catalog

,codigo,material,primeiro_ano,ultimo_ano,numero_anos,numero_registos
0,M01,Britas,2015,2026,12,144
1,M02,Areias,2015,2026,12,144
2,M03,Inertes,2015,2026,12,144
3,M04,Ladrilhos de calcário e granito,2015,2026,12,144
4,M05,Cantarias de calcário e granito,2015,2026,12,144
5,M06,Ladr. e cant. de calcário e granito,2015,2026,12,144
6,M07,Telhas cerâmicas,2015,2026,12,144
7,M08,Tijolos cerâmicos,2015,2026,12,144
8,M09,Produtos Cerâmicos Vermelhos,2015,2026,12,144
9,M10,Azulejos e Mosaicos,2015,2026,12,144


In [100]:
print(f"Materiais/códigos encontrados: {len(materials_catalog)}")

Materiais/códigos encontrados: 58


In [101]:
material_names = (
    materials_history
    .groupby("codigo")["material"]
    .nunique()
    .sort_values(ascending=False)
)

material_names[material_names > 1]

Series([], Name: material, dtype: int64)

In [102]:
materials_catalog = (
    materials_history
    [["codigo", "material"]]
    .drop_duplicates()
    .sort_values("codigo")
    .reset_index(drop=True)
)

materials_catalog.head()

,codigo,material
0,M01,Britas
1,M02,Areias
2,M03,Inertes
3,M04,Ladrilhos de calcário e granito
4,M05,Cantarias de calcário e granito


In [103]:
print(f"Materiais únicos encontrados: {len(materials_catalog)}")
materials_catalog

Materiais únicos encontrados: 58


,codigo,material
0,M01,Britas
1,M02,Areias
2,M03,Inertes
3,M04,Ladrilhos de calcário e granito
4,M05,Cantarias de calcário e granito
5,M06,Ladr. e cant. de calcário e granito
6,M07,Telhas cerâmicas
7,M08,Tijolos cerâmicos
8,M09,Produtos Cerâmicos Vermelhos
9,M10,Azulejos e Mosaicos


In [104]:
# ============================================================
# Validação do histórico de materiais
# ============================================================

codes_by_year = (
    materials_history
    .groupby("ano")["codigo"]
    .unique()
)

for ano, codes in codes_by_year.items():
    print(
        f"{ano}: {len(codes)} códigos | "
        f"{', '.join(sorted(codes))}"
    )

2015: 49 códigos | M01, M02, M03, M04, M05, M06, M07, M08, M09, M10, M12, M13, M14, M15, M16, M17, M18, M19, M20, M21, M22, M23, M24, M25, M26, M27, M28, M29, M30, M31, M32, M33, M34, M35, M36, M37, M39, M40, M41, M42, M43, M44, M45, M46, M47, M48, M49, M50, M51
2016: 49 códigos | M01, M02, M03, M04, M05, M06, M07, M08, M09, M10, M12, M13, M14, M15, M16, M17, M18, M19, M20, M21, M22, M23, M24, M25, M26, M27, M28, M29, M30, M31, M32, M33, M34, M35, M36, M37, M39, M40, M41, M42, M43, M44, M45, M46, M47, M48, M49, M50, M51
2017: 49 códigos | M01, M02, M03, M04, M05, M06, M07, M08, M09, M10, M12, M13, M14, M15, M16, M17, M18, M19, M20, M21, M22, M23, M24, M25, M26, M27, M28, M29, M30, M31, M32, M33, M34, M35, M36, M37, M39, M40, M41, M42, M43, M44, M45, M46, M47, M48, M49, M50, M51
2018: 49 códigos | M01, M02, M03, M04, M05, M06, M07, M08, M09, M10, M12, M13, M14, M15, M16, M17, M18, M19, M20, M21, M22, M23, M24, M25, M26, M27, M28, M29, M30, M31, M32, M33, M34, M35, M36, M37, M39, M40, M4

In [105]:
# Materiais que aparecem no histórico completo
materials_catalog = (
    materials_history[["codigo", "material"]]
    .drop_duplicates()
    .sort_values("codigo")
    .reset_index(drop=True)
)

print(f"Total de códigos encontrados: {len(materials_catalog)}")
materials_catalog

Total de códigos encontrados: 58


,codigo,material
0,M01,Britas
1,M02,Areias
2,M03,Inertes
3,M04,Ladrilhos de calcário e granito
4,M05,Cantarias de calcário e granito
5,M06,Ladr. e cant. de calcário e granito
6,M07,Telhas cerâmicas
7,M08,Tijolos cerâmicos
8,M09,Produtos Cerâmicos Vermelhos
9,M10,Azulejos e Mosaicos


In [106]:
m11 = materials_history[
    materials_history["codigo"]
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("M11")
]

print("Registos M11:", len(m11))
print("Anos em que aparece:", sorted(m11["ano"].unique()))

Registos M11: 0
Anos em que aparece: []


In [107]:
materials_catalog

,codigo,material
0,M01,Britas
1,M02,Areias
2,M03,Inertes
3,M04,Ladrilhos de calcário e granito
4,M05,Cantarias de calcário e granito
5,M06,Ladr. e cant. de calcário e granito
6,M07,Telhas cerâmicas
7,M08,Tijolos cerâmicos
8,M09,Produtos Cerâmicos Vermelhos
9,M10,Azulejos e Mosaicos


## Catálogo histórico de materiais

A partir dos ficheiros históricos disponibilizados pelo IMPIC foi construído um catálogo
único dos códigos de materiais encontrados ao longo do período analisado.

O catálogo mantém o código e a descrição do material, independentemente do número
de anos em que o material apresenta observações.

Este catálogo será utilizado posteriormente como referência para o dataset de previsão.

In [108]:
import json

materials_catalog_path = DATA_PROCESSED / "materials_catalog.json"

materials_catalog_records = (
    materials_catalog
    .rename(columns={
        "codigo": "codigo",
        "material": "descricao"
    })
    .to_dict(orient="records")
)

with open(materials_catalog_path, "w", encoding="utf-8") as f:
    json.dump(
        materials_catalog_records,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Catálogo gravado: {materials_catalog_path}")
print(f"Materiais: {len(materials_catalog_records)}")

Catálogo gravado: ..\data\processed\materials_catalog.json
Materiais: 58


## Influenciadores de custos

Nesta fase são registadas apenas as descrições dos influenciadores disponibilizados
pelas fontes utilizadas no projeto.

Não é estabelecida neste momento uma relação causal entre um influenciador e um
determinado material.

A associação entre materiais e influenciadores será objeto de análise numa fase
posterior do projeto, podendo posteriormente ser enriquecida no Anturio Construct.

In [109]:
influencers_catalog = pd.DataFrame(
    columns=[
        "codigo",
        "descricao"
    ]
)

influencers_catalog

,codigo,descricao


In [110]:
influencers_catalog_path = (
    DATA_PROCESSED / "influencers_catalog.json"
)

influencers_catalog.to_json(
    influencers_catalog_path,
    orient="records",
    force_ascii=False,
    indent=2
)

print(f"Catálogo de influenciadores: {influencers_catalog_path}")

Catálogo de influenciadores: ..\data\processed\influencers_catalog.json


## Validação da continuidade dos códigos

Durante a análise histórica foi verificado se todos os códigos de materiais
mantiveram presença nas diferentes publicações anuais.

A ausência de um código num determinado ficheiro não é interpretada
automaticamente como eliminação do material.

A análise deverá distinguir entre:

- código introduzido posteriormente;
- código que deixou de ser publicado;
- código ausente de determinada publicação;
- alteração da estrutura dos índices;
- erro ou incorreção numa publicação;
- republicação/correção de valores.

Esta distinção é particularmente importante para a construção do dataset
histórico utilizado na previsão.

In [111]:
# ============================================================
# Validação da presença dos códigos ao longo dos anos
# ============================================================

codes_by_year = (
    materials_history
    .groupby("ano")["codigo"]
    .unique()
)

for ano, codes in codes_by_year.items():
    print(
        f"{ano}: {len(codes):2d} códigos | "
        f"{', '.join(sorted(codes))}"
    )

2015: 49 códigos | M01, M02, M03, M04, M05, M06, M07, M08, M09, M10, M12, M13, M14, M15, M16, M17, M18, M19, M20, M21, M22, M23, M24, M25, M26, M27, M28, M29, M30, M31, M32, M33, M34, M35, M36, M37, M39, M40, M41, M42, M43, M44, M45, M46, M47, M48, M49, M50, M51
2016: 49 códigos | M01, M02, M03, M04, M05, M06, M07, M08, M09, M10, M12, M13, M14, M15, M16, M17, M18, M19, M20, M21, M22, M23, M24, M25, M26, M27, M28, M29, M30, M31, M32, M33, M34, M35, M36, M37, M39, M40, M41, M42, M43, M44, M45, M46, M47, M48, M49, M50, M51
2017: 49 códigos | M01, M02, M03, M04, M05, M06, M07, M08, M09, M10, M12, M13, M14, M15, M16, M17, M18, M19, M20, M21, M22, M23, M24, M25, M26, M27, M28, M29, M30, M31, M32, M33, M34, M35, M36, M37, M39, M40, M41, M42, M43, M44, M45, M46, M47, M48, M49, M50, M51
2018: 49 códigos | M01, M02, M03, M04, M05, M06, M07, M08, M09, M10, M12, M13, M14, M15, M16, M17, M18, M19, M20, M21, M22, M23, M24, M25, M26, M27, M28, M29, M30, M31, M32, M33, M34, M35, M36, M37, M39, M40, M4

In [112]:
code_presence = (
    materials_history
    .assign(presente=1)
    .pivot_table(
        index="ano",
        columns="codigo",
        values="presente",
        aggfunc="max",
        fill_value=0
    )
)

code_presence

codigo,M01,M02,M03,M04,M05,M06,M07,M08,M09,M10,M12,M13,M14,M15,M16,M17,M18,M19,M20,M21,M22,M23,M24,M25,M26,M27,M28,M29,M30,M31,M32,M33,M34,M35,M36,M37,M39,M40,M41,M42,M43,M44,M45,M46,M47,M48,M49,M50,M51,M52,M53,M54,M55,M56,M57,M58,M59,M60
ano,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2015,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0
2016,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0
2017,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0
2018,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0
2019,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0
2020,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0
2021,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0
2022,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0
2023,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0


In [113]:
first_year = (
    materials_history
    .groupby("codigo")["ano"]
    .min()
    .sort_values()
)

first_year

codigo
M01    2015
M02    2015
M03    2015
M04    2015
M05    2015
M06    2015
M07    2015
M08    2015
M09    2015
M10    2015
M12    2015
M13    2015
M14    2015
M15    2015
M16    2015
M17    2015
M18    2015
M19    2015
M20    2015
M21    2015
M22    2015
M23    2015
M24    2015
M25    2015
M26    2015
M27    2015
M28    2015
M29    2015
M30    2015
M31    2015
M32    2015
M33    2015
M34    2015
M35    2015
M36    2015
M37    2015
M39    2015
M40    2015
M41    2015
M42    2015
M43    2015
M44    2015
M45    2015
M46    2015
M47    2015
M48    2015
M49    2015
M50    2015
M51    2015
M52    2020
M53    2020
M54    2020
M55    2020
M56    2020
M57    2020
M58    2022
M59    2026
M60    2026
Name: ano, dtype: int64

In [114]:
last_year = (
    materials_history
    .groupby("codigo")["ano"]
    .max()
    .sort_values()
)

last_year

codigo
M01    2026
M02    2026
M03    2026
M04    2026
M05    2026
M06    2026
M07    2026
M08    2026
M09    2026
M10    2026
M12    2026
M13    2026
M14    2026
M15    2026
M16    2026
M17    2026
M18    2026
M19    2026
M20    2026
M21    2026
M22    2026
M23    2026
M24    2026
M25    2026
M26    2026
M27    2026
M28    2026
M29    2026
M30    2026
M31    2026
M32    2026
M33    2026
M34    2026
M35    2026
M36    2026
M37    2026
M39    2026
M40    2026
M41    2026
M42    2026
M43    2026
M44    2026
M45    2026
M46    2026
M47    2026
M48    2026
M49    2026
M50    2026
M51    2026
M52    2026
M53    2026
M54    2026
M55    2026
M56    2026
M57    2026
M58    2026
M59    2026
M60    2026
Name: ano, dtype: int64

In [115]:
special_codes = ["M11", "M38"]

materials_history[
    materials_history["codigo"].isin(special_codes)
].groupby("codigo").agg(
    primeiro_ano=("ano", "min"),
    ultimo_ano=("ano", "max"),
    observacoes=("codigo", "size")
)

,primeiro_ano,ultimo_ano,observacoes
codigo,,,


In [116]:
# ============================================================
# Inspeção dos ficheiros GLOBAIS
# ============================================================

global_local_files = sorted(DATA_RAW.glob("cife*.xls"))

print(f"Ficheiros globais encontrados: {len(global_local_files)}")

for filepath in global_local_files:
    print(filepath.name)

Ficheiros globais encontrados: 34
cife2010_.xls
cife2010_4%20MATERIAIS.xls
cife2011_.xls
cife2011_4%20MATERIAIS.xls
cife2012_.xls
cife2012_4%20MATERIAIS.xls
cife2013_.xls
cife2013_4%20MATERIAIS.xls
cife2014_.xls
cife2014_4%20MATERIAIS.xls
cife2015_.xls
cife2015_MATERIAIS.xls
cife2016_.xls
cife2016_MATERIAIS.xls
cife2017_.xls
cife2017_MATERIAIS.xls
cife2018_.xls
cife2018_MATERIAIS.xls
cife2019_.xls
cife2019_MATERIAIS.xls
cife2020_.xls
cife2020_MATERIAIS.xls
cife2021_.xls
cife2021_MATERIAIS.xls
cife2022_.xls
cife2022_MATERIAIS.xls
cife2023_.xls
cife2023_MATERIAIS.xls
cife2024_.xls
cife2024_MATERIAIS.xls
cife2025_.xls
cife2025_MATERIAIS.xls
cife2026_.xls
cife2026_MATERIAIS.xls


In [117]:
print("Materiais:", len(material_files))
print("Globais (URLs):", len(global_urls))
print("Ficheiros locais:", len(global_local_files))

Materiais: 12
Globais (URLs): 22
Ficheiros locais: 34


In [118]:
for filepath in global_local_files:
    print(filepath.name)

cife2010_.xls
cife2010_4%20MATERIAIS.xls
cife2011_.xls
cife2011_4%20MATERIAIS.xls
cife2012_.xls
cife2012_4%20MATERIAIS.xls
cife2013_.xls
cife2013_4%20MATERIAIS.xls
cife2014_.xls
cife2014_4%20MATERIAIS.xls
cife2015_.xls
cife2015_MATERIAIS.xls
cife2016_.xls
cife2016_MATERIAIS.xls
cife2017_.xls
cife2017_MATERIAIS.xls
cife2018_.xls
cife2018_MATERIAIS.xls
cife2019_.xls
cife2019_MATERIAIS.xls
cife2020_.xls
cife2020_MATERIAIS.xls
cife2021_.xls
cife2021_MATERIAIS.xls
cife2022_.xls
cife2022_MATERIAIS.xls
cife2023_.xls
cife2023_MATERIAIS.xls
cife2024_.xls
cife2024_MATERIAIS.xls
cife2025_.xls
cife2025_MATERIAIS.xls
cife2026_.xls
cife2026_MATERIAIS.xls


In [119]:
from urllib.parse import unquote
from pathlib import Path

filename = Path(url).name

filename = Path(unquote(url)).name

url = "https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2014_%20MATERIAIS.xls"

filename = Path(unquote(url)).name

print(filename)

cife2014_ MATERIAIS.xls


In [120]:
global_local_files = sorted(DATA_RAW.glob("cife*.xls"))

for filepath in global_local_files:
    print("\n" + "=" * 60)
    print(filepath.name)

    sheets = pd.ExcelFile(filepath).sheet_names
    print("Folhas:", sheets)


cife2010_.xls
Folhas: ['2010']

cife2010_4%20MATERIAIS.xls
Folhas: ['2010-Mat-base100 Dez91;Jan04']

cife2011_.xls
Folhas: ['2011']

cife2011_4%20MATERIAIS.xls
Folhas: ['2011-Mat-base100 Dez91;Jan04']

cife2012_.xls
Folhas: ['2012']

cife2012_4%20MATERIAIS.xls
Folhas: ['2012-Mat-base100 Dez91;Jan04']

cife2013_.xls
Folhas: ['2013']

cife2013_4%20MATERIAIS.xls
Folhas: ['2013-Mat-base100 Dez91;Jan04']

cife2014_.xls
Folhas: ['2014']

cife2014_4%20MATERIAIS.xls
Folhas: ['2014-Mat-base100 Dez91;Jan04']

cife2015_.xls
Folhas: ['2015']

cife2015_MATERIAIS.xls
Folhas: ['2015-Mat-base100 Dez91;Jan04']

cife2016_.xls
Folhas: ['2016']

cife2016_MATERIAIS.xls
Folhas: ['2016-Mat-base100 Dez91;Jan04']

cife2017_.xls
Folhas: ['2017']

cife2017_MATERIAIS.xls
Folhas: ['2017-Mat-base100 Dez91;Jan04']

cife2018_.xls
Folhas: ['2018']

cife2018_MATERIAIS.xls
Folhas: ['2018-Mat-base100 Dez91;Jan04']

cife2019_.xls
Folhas: ['2019']

cife2019_MATERIAIS.xls
Folhas: ['2019-Mat-base100 Dez91;Jan04']

cife2020_

In [121]:
filepath = DATA_RAW / "cife2026_.xls"

xls = pd.ExcelFile(filepath)

print(xls.sheet_names)

for sheet in xls.sheet_names:
    df = pd.read_excel(
        filepath,
        sheet_name=sheet,
        header=None
    )

    print("\n---", sheet, "---")
    print(df.head(15))

['2026']

--- 2026 ---
                                  0                                      1            2      3      4      5      6      7    8    9    10   11   12   13
0                           QUADRO I                                    NaN          NaN    NaN    NaN    NaN    NaN    NaN  NaN  NaN  NaN  NaN  NaN  NaN
1   ÍNDICES DE CUSTOS DE MÃO DE OBRA                                    NaN          NaN    NaN    NaN    NaN    NaN    NaN  NaN  NaN  NaN  NaN  NaN  NaN
2          BASE 100: JANEIRO DE 2004                                    NaN          NaN    NaN    NaN    NaN    NaN    NaN  NaN  NaN  NaN  NaN  NaN  NaN
3                            ÍNDICES                                    NaN  ANO DE 2026    NaN    NaN    NaN    NaN    NaN  NaN  NaN  NaN  NaN  NaN  NaN
4                                NaN                                    NaN          JAN    FEV    MAR    ABR    MAI    JUN  JUL  AGO  SET  OUT  NOV  DEZ
5                             Global                 

In [122]:
filepath = DATA_RAW / "cife2026_.xls"

df_global = pd.read_excel(
    filepath,
    sheet_name="2026",
    header=None
)

print("Dimensão:", df_global.shape)

# Mostrar linhas onde aparece algo parecido com F01, F02, F03...
for idx, row in df_global.iterrows():
    valores = row.dropna().astype(str).tolist()

    if any(
        valor.strip().upper().startswith("F")
        and valor.strip()[1:].isdigit()
        for valor in valores
    ):
        print(idx, valores)

Dimensão: (134, 14)
7 ['F01', '  Edifícios de habitação ', '230.7', '230.7', '230.7', '236.5', '236.5', '236.5']
8 ['F02', '  Edifícios administrativos ', '231.2', '231.2', '231.2', '237.3', '237.3', '237.3']
9 ['F03', '  Edifícios escolares ', '231.7', '231.7', '231.7', '237.8', '237.8', '237.8']
10 ['F04', '  Edifícios para o sector da saúde ', '231.5', '231.5', '231.5', '237.3', '237.3', '237.3']
11 ['F05', '  Reabilitação ligeira de edifícios ', '226.6', '226.6', '226.6', '234.2', '234.2', '234.2']
12 ['F06', '  Reabilitação média de edifícios ', '226.7', '226.7', '226.7', '233.9', '233.9', '233.9']
13 ['F07', '  Reabilitação profunda de edifícios ', '226', '226', '226', '232.6', '232.6', '232.6']
14 ['F08', '  Campos de jogos com balneários ', '232.2', '232.2', '232.2', '238.9', '238.9', '238.9']
15 ['F09', '  Arranjos exteriores ', '232.8', '232.8', '232.8', '239.3', '239.3', '239.3']
16 ['F10', '  Estradas ', '220.5', '220.5', '220.5', '228', '228', '228']
17 ['F11', '  Túneis '

In [123]:
# Mostrar as primeiras 100 linhas sem truncar as colunas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)

df_global.iloc[:100]

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,QUADRO I,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ÍNDICES DE CUSTOS DE MÃO DE OBRA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BASE 100: JANEIRO DE 2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ÍNDICES,NaN,ANO DE 2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,JAN,FEV,MAR,ABR,MAI,JUN,JUL,AGO,SET,OUT,NOV,DEZ
5,Global,NaN,225.2,225.2,225.2,230.6,230.6,230.6,NaN,NaN,NaN,NaN,NaN,NaN
6,Código,Por fórmula tipo (*),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,F01,Edifícios de habitação,230.7,230.7,230.7,236.5,236.5,236.5,NaN,NaN,NaN,NaN,NaN,NaN
8,F02,Edifícios administrativos,231.2,231.2,231.2,237.3,237.3,237.3,NaN,NaN,NaN,NaN,NaN,NaN
9,F03,Edifícios escolares,231.7,231.7,231.7,237.8,237.8,237.8,NaN,NaN,NaN,NaN,NaN,NaN


In [124]:
print(type(global_urls))
print(global_urls[:3])

<class 'list'>
['https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2026_.xls', 'https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2025_.xls', 'https://www.impic.pt/impic/assets/misc/indicesCIFE/cife2024_.xls']


In [125]:
from pathlib import Path
import requests

print(f"Download de {len(global_urls)} ficheiros globais...")

for i, url in enumerate(global_urls, start=1):

    filename = Path(url).name

    # normalizar o nome
    filename = filename.replace("%20", " ")
    filename = filename.replace(" ", "_")

    filepath = DATA_RAW / filename

    print(f"{i:02d}/{len(global_urls)} - {filename}")

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    filepath.write_bytes(response.content)

print("\nDownload dos ficheiros globais concluído.")

Download de 22 ficheiros globais...
01/22 - cife2026_.xls
02/22 - cife2025_.xls
03/22 - cife2024_.xls
04/22 - cife2023_.xls
05/22 - cife2022_.xls
06/22 - cife2021_.xls
07/22 - cife2020_.xls
08/22 - cife2019_.xls
09/22 - cife2018_.xls
10/22 - cife2017_.xls
11/22 - cife2016_.xls
12/22 - cife2015_.xls
13/22 - cife2014_4_MATERIAIS.xls
14/22 - cife2014_.xls
15/22 - cife2013_4_MATERIAIS.xls
16/22 - cife2013_.xls
17/22 - cife2012_4_MATERIAIS.xls
18/22 - cife2012_.xls
19/22 - cife2011_4_MATERIAIS.xls
20/22 - cife2011_.xls
21/22 - cife2010_4_MATERIAIS.xls
22/22 - cife2010_.xls

Download dos ficheiros globais concluído.


In [126]:
materials_history

,ano,codigo,material,mes,indice,ficheiro
0,2015,M01,Britas,JAN,104.4,cife2015_MATERIAIS.xls
1,2015,M01,Britas,FEV,104.7,cife2015_MATERIAIS.xls
2,2015,M01,Britas,MAR,105.0,cife2015_MATERIAIS.xls
3,2015,M01,Britas,ABR,104.8,cife2015_MATERIAIS.xls
4,2015,M01,Britas,MAI,104.5,cife2015_MATERIAIS.xls
...,...,...,...,...,...,...
7639,2026,M60,Janelas e portas de PVC,AGO,NaN,cife2026_MATERIAIS.xls
7640,2026,M60,Janelas e portas de PVC,SET,NaN,cife2026_MATERIAIS.xls
7641,2026,M60,Janelas e portas de PVC,OUT,NaN,cife2026_MATERIAIS.xls
7642,2026,M60,Janelas e portas de PVC,NOV,NaN,cife2026_MATERIAIS.xls


In [127]:
materials_index_path = DATA_PROCESSED / "materials_index.csv"

materials_history.to_csv(
    materials_index_path,
    index=False,
    encoding="utf-8"
)

print(f"Índice de materiais guardado em:")
print(materials_index_path)

print(f"\nRegistos: {len(materials_history):,}")
print(f"Materiais: {materials_history['codigo'].nunique()}")
print(f"Anos: {materials_history['ano'].min()} - {materials_history['ano'].max()}")

Índice de materiais guardado em:
..\data\processed\materials_index.csv

Registos: 7,644
Materiais: 58
Anos: 2015 - 2026


In [128]:
materials_history.head()

,ano,codigo,material,mes,indice,ficheiro
0,2015,M01,Britas,JAN,104.4,cife2015_MATERIAIS.xls
1,2015,M01,Britas,FEV,104.7,cife2015_MATERIAIS.xls
2,2015,M01,Britas,MAR,105.0,cife2015_MATERIAIS.xls
3,2015,M01,Britas,ABR,104.8,cife2015_MATERIAIS.xls
4,2015,M01,Britas,MAI,104.5,cife2015_MATERIAIS.xls


In [129]:
print("=== RESUMO DA RECOLHA ===")

print(f"Registos: {len(materials_history):,}")
print(f"Materiais: {materials_history['codigo'].nunique()}")
print(f"Ano inicial: {materials_history['ano'].min()}")
print(f"Ano final: {materials_history['ano'].max()}")

print("\nRegistos por ano:")
print(
    materials_history
    .groupby("ano")
    .size()
    .to_string()
)

print("\nMateriais por ano:")
print(
    materials_history
    .groupby("ano")["codigo"]
    .nunique()
    .to_string()
)

=== RESUMO DA RECOLHA ===
Registos: 7,644
Materiais: 58
Ano inicial: 2015
Ano final: 2026

Registos por ano:
ano
2015    588
2016    588
2017    588
2018    588
2019    588
2020    660
2021    660
2022    672
2023    672
2024    672
2025    672
2026    696

Materiais por ano:
ano
2015    49
2016    49
2017    49
2018    49
2019    49
2020    55
2021    55
2022    56
2023    56
2024    56
2025    56
2026    58


In [130]:
from pathlib import Path

DATA_RAW = Path("../data/raw/impic")

files = sorted(DATA_RAW.glob("*.xls"))

print(f"Ficheiros encontrados: {len(files)}")

for f in files:
    print(f.name)

Ficheiros encontrados: 34
cife2010_.xls
cife2010_4_MATERIAIS.xls
cife2011_.xls
cife2011_4_MATERIAIS.xls
cife2012_.xls
cife2012_4_MATERIAIS.xls
cife2013_.xls
cife2013_4_MATERIAIS.xls
cife2014_.xls
cife2014_4_MATERIAIS.xls
cife2015_.xls
cife2015_MATERIAIS.xls
cife2016_.xls
cife2016_MATERIAIS.xls
cife2017_.xls
cife2017_MATERIAIS.xls
cife2018_.xls
cife2018_MATERIAIS.xls
cife2019_.xls
cife2019_MATERIAIS.xls
cife2020_.xls
cife2020_MATERIAIS.xls
cife2021_.xls
cife2021_MATERIAIS.xls
cife2022_.xls
cife2022_MATERIAIS.xls
cife2023_.xls
cife2023_MATERIAIS.xls
cife2024_.xls
cife2024_MATERIAIS.xls
cife2025_.xls
cife2025_MATERIAIS.xls
cife2026_.xls
cife2026_MATERIAIS.xls


In [131]:
global_local_files = sorted(DATA_RAW.glob("cife*_*.xls"))

print(f"Ficheiros encontrados: {len(global_local_files)}")

Ficheiros encontrados: 34


In [132]:
globais = sorted(
    p for p in DATA_RAW.glob("cife*.xls")
    if "_MATERIAIS" not in p.name
)

materiais = sorted(
    p for p in DATA_RAW.glob("cife*_MATERIAIS.xls")
)

print(f"Ficheiros globais:   {len(globais)}")
print(f"Ficheiros materiais: {len(materiais)}")

print("\nGlobais:")
for p in globais:
    print(p.name)

print("\nMateriais:")
for p in materiais:
    print(p.name)

Ficheiros globais:   17
Ficheiros materiais: 17

Globais:
cife2010_.xls
cife2011_.xls
cife2012_.xls
cife2013_.xls
cife2014_.xls
cife2015_.xls
cife2016_.xls
cife2017_.xls
cife2018_.xls
cife2019_.xls
cife2020_.xls
cife2021_.xls
cife2022_.xls
cife2023_.xls
cife2024_.xls
cife2025_.xls
cife2026_.xls

Materiais:
cife2010_4_MATERIAIS.xls
cife2011_4_MATERIAIS.xls
cife2012_4_MATERIAIS.xls
cife2013_4_MATERIAIS.xls
cife2014_4_MATERIAIS.xls
cife2015_MATERIAIS.xls
cife2016_MATERIAIS.xls
cife2017_MATERIAIS.xls
cife2018_MATERIAIS.xls
cife2019_MATERIAIS.xls
cife2020_MATERIAIS.xls
cife2021_MATERIAIS.xls
cife2022_MATERIAIS.xls
cife2023_MATERIAIS.xls
cife2024_MATERIAIS.xls
cife2025_MATERIAIS.xls
cife2026_MATERIAIS.xls
